In [1]:
import os
import cv2
import time
import queue
import threading
import keyboard
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from datetime import datetime


In [2]:
# Define parameters for movement control
MIN_SIZE = 50  # Minimum pixel size to consider as an obstacle
THRESHOLD_DISTANCE = 0.5  # Threshold in meters for obstacle proximity
SPEED = 1.0  # Base speed of the drone
reward_count = 0  # Count successful avoidance
experience_data = []  # Store experiences for retraining
experience_threshold = 100
success_threshold = 5  # Number of successful avoidance actions before retraining

In [3]:
frame_queue = queue.Queue(maxsize=1)

In [4]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, concatenate
from tensorflow.keras.models import Model


In [5]:
print(cv2.__version__)

4.9.0


In [6]:
def build_unet(input_shape=(128, 128, 3)):
    """
    Build a U-Net model for depth estimation.
    """
    inputs = Input(shape=input_shape)

    # Encoder: Contracting Path
    c1 = Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    c1 = Conv2D(64, (3, 3), activation='relu', padding='same')(c1)
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = Conv2D(128, (3, 3), activation='relu', padding='same')(p1)
    c2 = Conv2D(128, (3, 3), activation='relu', padding='same')(c2)
    p2 = MaxPooling2D((2, 2))(c2)

    c3 = Conv2D(256, (3, 3), activation='relu', padding='same')(p2)
    c3 = Conv2D(256, (3, 3), activation='relu', padding='same')(c3)
    p3 = MaxPooling2D((2, 2))(c3)

    # Bottleneck
    c4 = Conv2D(512, (3, 3), activation='relu', padding='same')(p3)
    c4 = Conv2D(512, (3, 3), activation='relu', padding='same')(c4)

    # Decoder: Expanding Path
    u5 = Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(c4)
    u5 = concatenate([u5, c3])
    c5 = Conv2D(256, (3, 3), activation='relu', padding='same')(u5)
    c5 = Conv2D(256, (3, 3), activation='relu', padding='same')(c5)

    u6 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = concatenate([u6, c2])
    c6 = Conv2D(128, (3, 3), activation='relu', padding='same')(u6)
    c6 = Conv2D(128, (3, 3), activation='relu', padding='same')(c6)

    u7 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = concatenate([u7, c1])
    c7 = Conv2D(64, (3, 3), activation='relu', padding='same')(u7)
    c7 = Conv2D(64, (3, 3), activation='relu', padding='same')(c7)

    # Output Layer for Depth Map
    outputs = Conv2D(1, (1, 1), activation='linear')(c7)

    model = Model(inputs=[inputs], outputs=[outputs])
    return model

In [7]:
# Create and compile the U-Net model
unet_model = build_unet()
unet_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [8]:
# Example placeholders for dataset loading
X_train = np.random.rand(1000, 128, 128, 3)  # 1000 RGB training images
Y_train = np.random.rand(1000, 128, 128, 1)  # 1000 depth maps
X_val = np.random.rand(200, 128, 128, 3)     # 200 RGB validation images
Y_val = np.random.rand(200, 128, 128, 1)     # 200 depth maps

In [9]:
# Training the model
history = unet_model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=10,               # Start with 10 epochs, adjust based on results
    batch_size=16,
)

Epoch 1/10
 8/63 ━━━━━━━━━━━━━━━━━━━━ 2:38 3s/step - loss: 0.4782 - mae: 0.5237

KeyboardInterrupt: 

In [ ]:
# Plot training & validation loss values
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss Over Epochs')
plt.show()

In [ ]:
# Save initial model
unet_model.save(f'depth_estimation_model_{datetime.now().strftime("%Y%m%d_%H%M%S")}.h5')

In [ ]:
#Preprocess function for real-time frames
def preprocess_frame(frame):
    frame_resized = cv2.resize(frame, (128, 128))
    frame_normalized = frame_resized / 255.0
    return np.expand_dims(frame_normalized, axis=0)

In [ ]:
def calculate_obstacle_size(predicted_depth):
    return np.sum(predicted_depth < THRESHOLD_DISTANCE)

In [ ]:
def calculate_distance_to_nearest_obstacle(predicted_depth):
    return np.min(predicted_depth)

In [ ]:
#Drone control
def control_drone(direction):
    print(f"Moving {direction}")

In [ ]:
def retrain_model():
    if len(experience_data) == 0:
        print("No experience data to retrain.")
        return
    
    X_new = np.array([cv2.resize(exp[0], (128, 128)) / 255.0 for exp in experience_data])
    Y_new = np.array([exp[1] for exp in experience_data])
    
    # Adding validation data during retraining
    history = unet_model.fit(X_new, Y_new, validation_data=(X_val, Y_val), epochs=5, batch_size=16, verbose=1)

    # Save if validation improves
    if 'val_loss' in history.history and history.history['loss'][-1] < np.min(history.history['val_loss']):
        model_filename = f'depth_estimation_model_updated_{datetime.now().strftime("%Y%m%d_%H%M%S")}.h5'
        unet_model.save(model_filename)
        print(f"Model saved as {model_filename}")

    experience_data.clear()


In [ ]:
def collect_experience(frame, depth, command, outcome):
    experience_data.append((frame, depth, command, outcome))
    if len(experience_data) >= experience_threshold:
        retrain_model()

In [ ]:
def capture_frames():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Failed to open camera.")
        return

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Failed to capture frame.")
            break
        
        small_frame = cv2.resize(frame, (128, 128))
        
        if not frame_queue.full():
            frame_queue.put((frame, small_frame))
        
        time.sleep(0.03)  # Capture roughly 30 frames per second

    cap.release()
    cv2.destroyAllWindows()


In [ ]:
def process_and_display_frames():
    global reward_count
    
    while True:
        if frame_queue.empty():
            continue
        
        original_frame, small_frame = frame_queue.get()

        preprocessed = preprocess_frame(small_frame)
        predicted_depth = unet_model.predict(preprocessed).reshape(128, 128)
        
        obstacle_size = calculate_obstacle_size(predicted_depth)
        nearest_distance = calculate_distance_to_nearest_obstacle(predicted_depth)

        if obstacle_size > MIN_SIZE:
            command = "backward" if nearest_distance < THRESHOLD_DISTANCE else "right"
            control_drone(command)
            reward_count += 1
            collect_experience(original_frame, predicted_depth, command, 1)
            
            if reward_count >= success_threshold:
                retrain_model()
                reward_count = 0
        else:
            command = "forward"
            control_drone(command)

        overlay_text = (f"Command: {command}, Nearest Distance: {nearest_distance:.2f}, "
                        f"Obstacle Size: {obstacle_size}")
        cv2.putText(original_frame, overlay_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.imshow("Drone Interface", original_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Process terminated.")
            break


In [ ]:
capture_thread = threading.Thread(target=capture_frames)
display_thread = threading.Thread(target=process_and_display_frames)

capture_thread.start()
display_thread.start()

capture_thread.join()
display_thread.join()

cv2.destroyAllWindows()
